# M2 v2 · Evaluación en prueba (una sola vez)

**TFM · HAM10000** · Responsable: **Alex Reinoso**. GPU T4.

Evalúa, una sola vez, las tres semillas con alias `final` (2024). Para cada red: argmax y `p(mel) ≥ umbral_mel`. Registra `test_*`, `test_umbral_*`, `confusion_test_norm.png`, `confusion_test_umbral_norm.png`, `predicciones_test.csv` y `falsos_negativos_mel.csv`. Al terminar pone `test_evaluado=true`. Si ese tag ya está, se niega a repetir.

No cambia el modelo, el umbral ni la selección. Secretos `DAGSHUB_USER` y `DAGSHUB_TOKEN`.


In [ ]:
BRANCH = "m2/seleccion-v2"
AUTOR = "alex"
CONFIRMO_TEST = True
ARCHS = ["resnet50", "densenet121", "efficientnetb0"]


## 1. Sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU"
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}
!git log --oneline -1
!pip install -q -r requirements.txt


In [ ]:
import os, shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
assert CONFIRMO_TEST, "CONFIRMO_TEST está en False. No se abre el conjunto de prueba."
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not (LOCAL / "images").exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido ({time.time()-t0:.0f} s)")
assert len(list((LOCAL / "images").glob("*.jpg"))) == 10015
assert file_sha256("data/splits/split_v1.csv") == "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
print("[+] Split v1 verificado")


In [ ]:
import os
from google.colab import userdata
from src.tracking import setup_mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print(setup_mlflow(cfg))


## 2. Evaluar

Primero se guarda el umbral con todos sus decimales. El valor de seis decimales ya anotado redondea hacia arriba y, en DenseNet121 y EfficientNetB0, deja la sensibilidad de melanoma en validación por debajo de 0,80. No es un umbral nuevo.

Después, una arquitectura cada vez. Si una ya está marcada, el bucle se detiene y no toca las demás.

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
UMBRAL_PRECISO = {
    "c009467e38f04b22a0208f6732c58e1a": "0.12729512155056",
    "a16c233b18b043f49a84e5afed0cf3ab": "0.08336453884840012",
    "de11d3148353403ba1c091e6b66f0abf": "0.1987808346748352",
}
for run_id, t in UMBRAL_PRECISO.items():
    if "umbral_mel_preciso" not in client.get_run(run_id).data.params:
        client.log_param(run_id, "umbral_mel_preciso", t)
    print(run_id, client.get_run(run_id).data.params["umbral_mel_preciso"])


In [ ]:
assert CONFIRMO_TEST
from src.evaluation import lanzar_evaluacion_test
from src.metrics import CLASES
resultados = []
for arch in ARCHS:
    print(f"\n{'='*20} {arch} {'='*20}")
    r = lanzar_evaluacion_test(arch, cfg, autor=AUTOR)
    resultados.append(r)
    print(f"[+] run={r['run_id']} modelo={r['run_id_modelo']} umbral={r['umbral_mel']:.6f}")
    print(f"    FN melanoma={r['fn_total']} recuperados={r['fn_recuperados']} {r['fn_por_clase']}")


In [ ]:
def fmt(x):
    return f"{x:.3f}"

lineas = ["# Evaluación en prueba v2", ""]
for r in resultados:
    m, u = r["metricas"], r["metricas_umbral"]
    lineas += [
        f"## {r['arch']}",
        f"Run de prueba: `{r['run_id']}`. Modelo: `{r['run_id_modelo']}`. umbral_mel={r['umbral_mel']:.6f}.",
        "Artefactos: `confusion_test_norm.png`, `confusion_test_umbral_norm.png`, `predicciones_test.csv`, `falsos_negativos_mel.csv`.",
        "",
        "| Criterio | macro_f1 | balanced_accuracy | auc_macro_ovr | mel_sens | mel_prec | mel_spec | min_sens | clase mínima |",
        "|---|---|---|---|---|---|---|---|---|",
        f"| argmax | {fmt(m['macro_f1'])} | {fmt(m['balanced_accuracy'])} | {fmt(m['auc_macro_ovr'])} | {fmt(m['mel_sensitivity'])} | {fmt(m['mel_precision'])} | {fmt(m['specificity_mel'])} | {fmt(m['min_class_sensitivity'])} | {r['clase_minima']} |",
        f"| umbral | {fmt(u['macro_f1'])} | {fmt(u['balanced_accuracy'])} | {fmt(u['auc_macro_ovr'])} | {fmt(u['mel_sensitivity'])} | {fmt(u['mel_precision'])} | {fmt(u['specificity_mel'])} | {fmt(u['min_class_sensitivity'])} | {r['clase_minima_umbral']} |",
        "",
        "| Clase | sens | prec | spec | auc | ap | sens umbral | prec umbral | spec umbral |",
        "|---|---|---|---|---|---|---|---|---|",
    ]
    for c in CLASES:
        lineas.append(
            f"| {c} | {fmt(m[f'sensitivity_{c}'])} | {fmt(m[f'precision_{c}'])} | {fmt(m[f'specificity_{c}'])} | {fmt(m[f'auc_{c}'])} | {fmt(m[f'ap_{c}'])} | {fmt(u[f'sensitivity_{c}'])} | {fmt(u[f'precision_{c}'])} | {fmt(u[f'specificity_{c}'])} |"
        )
    lineas += [
        "",
        f"Falsos negativos de melanoma (argmax): {r['fn_total']}. Se van a: {r['fn_por_clase']}. Recuperados con el umbral: {r['fn_recuperados']}.",
        "",
    ]
texto = "\n".join(lineas)
print(texto)
Path("/content/drive/MyDrive/TFM/evaluacion_test_v2.md").write_text(texto)
print("Guardado en Mi unidad/TFM/evaluacion_test_v2.md")
